# מעבדה 12 — פונקציית החלוקה

במעבדה הזו תמסרו למחשב את הלוגריתם של פונקציית החלוקה — מספר אחד לכל טמפרטורה — ותבקשו ממנו
את האנרגיה הממוצעת, את האנטרופיה, את האנרגיה החופשית ואת קיבול החום. הוא אינו יודע דבר נוסף על
המערכת: לא את הרמות, לא את ההסתברויות, לא את הצורות הסגורות. אחר כך תבדקו כל תשובה מול הצורות
הסגורות, ותגלו עד כמה אפשר לסמוך על הנגזרות הנומריות.

המסלול: המודל והאובייקטים שלו (חלק 1); פונקציית החלוקה ביד (חלק 2); מכונת ה-Z על שלוש מערכות,
והאנרגיה החופשית נבחנת מול זו של מודול 10 (חלק 3); כיצד צעד ההפרש שולט בשגיאה (חלק 4); הפרמגנט —
חוק קירי והפלוקטואציות שלו (חלק 5); הפרמגנט באנרגיה קבועה, טמפרטורות שליליות ולאן זורם החום
(חלק 6); המתנד ההרמוני, חלוקת האנרגיה השווה והקפיאה (חלק 7); הסימולציה של מודול 01 תחת פונקציית
החלוקה של המתנד (חלק 8); הבדיקות האוטומטיות (חלק 9); וחקירה חופשית (חלק 10).

עברו עליה בסדר. במקום שבו המחברת מבקשת מכם לנבא, רשמו את ניבויכם בתא המיועד לכך **לפני**
הרצת התא הבא. ניבוי שהתחייבתם אליו הוא הדרך האמינה היחידה לגלות שטעיתם.

## מפרט המודל

| | |
|---|---|
| **מערכת** | מערכות בעלות רמות בדידות נתונות: מערכת דו-רמתית (פער $\Delta$); $N$ מומנטים בלתי תלויים $\mu$ בשדה $B$, באנרגיות $\mp\mu B$; סולם הרמוני $\hbar\omega(n + 1/2)$ |
| **דינמיקה** | אין: ממוצעים קנוניים מצורות סגורות או מסכומים מפורשים; רק חלק 8 יורש מספרים אקראיים, מן הסימולציה של מודול 01 |
| **גבול** | מגע עם אמבט אינסופי בטמפרטורה $T$; הרמות קבועות, ולכן שום עבודה אינה נעשית וכל שינוי של $U$ הוא חום |
| **צבר** | קנוני, בדיוק: $P_s = e^{-\beta E_s}/Z$ |
| **מוזנח** | אינטראקציות בין תת-מערכות, הזזות של רמות מעבר לאיבר הלינארי בשדה, החלפה של חלקיקים זהים |
| **תקף כאשר** | הרמות קבועות וידועות, ותת-המערכות בלתי תלויות, ולכן $Z_N = z^N$ |
| **אופני כישלון** | מומנטים באינטראקציה (מודול 15); חלקיקים זהים בצפיפות גבוהה או בטמפרטורה $T$ נמוכה (מודול 17); רמות הזזות עם משתני המצב |

In [ ]:
# JupyterLite runs this notebook in the browser, where the course package and a few pure-
# Python libraries have to be installed into the kernel first. Under a local Jupyter they
# are already importable and this whole cell does nothing.
#
# thermolab is installed without its dependency graph on purpose: Pyodide supplies its own
# builds of numpy, scipy, matplotlib and sympy, older than the versions resolved for the
# development environment, and asking for those floors would send the installer to PyPI for
# packages that have no WebAssembly wheels. Add any new *pure-Python* dependency of
# thermolab to the list below.
try:
    import piplite
except ImportError:
    pass
else:
    await piplite.install(["pint", "ipywidgets", "jupyterquiz"])
    await piplite.install("thermolab", deps=False)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from thermolab import equilibrium, partition
from thermolab.constants import K_B
from thermolab.validation import (
    convergence_study,
    relative_error,
    scaling_exponent,
    seed_study,
)

# partition.py draws no random numbers: it evaluates sums and closed forms. Part 8 runs module
# 01's simulation, which does, and every generator there is passed in explicitly.

QUANTUM = 1.0e-21  # J: the two-level gap and the oscillator's hbar*omega
MU = 9.274e-24  # J/T: one Bohr magneton, the moment of an electron spin
FIELD = 1.0  # T
print(f"quantum / k_B = {QUANTUM / K_B:.2f} K")
print(f"mu B / k_B    = {MU * FIELD / K_B:.4f} K")

## חלק 1 — האובייקטים

כל פונקציית חלוקה במודול `thermolab.partition` מוחזרת בצורת **הלוגריתם** שלה. עבור עשרת אלפים
מומנטים, $Z_N$ הוא בערך $2^{10000}$ — שום מספר נקודה צפה אינו יכול להכיל אותו — ואילו $\ln Z_N$
הוא מספר צנוע, וכל גודל תרמודינמי דורש את $\ln Z$ בלבד.

| פונקציה | מערכת | $\ln Z$ |
|---|---|---|
| `log_z_two_level(gap, T)` | רמות $0$, $\Delta$ | $\ln(1 + e^{-\beta\Delta})$ |
| `log_z_paramagnet(N, mu, B, T)` | $N$ מומנטים באנרגיות $\mp\mu B$ | $N\ln(2\cosh\beta\mu B)$ |
| `log_z_harmonic(hbar_omega, T)` | סולם $\hbar\omega(n + 1/2)$ | $-x/2 - \ln(1 - e^{-x})$ |

ומכונה אחת, `thermo_from_z(log_z, temperatures)`, שמקבלת רק פונקציה המחזירה את
$\ln Z$ ומחזירה את $U$, $\sigma_E^2$, $C$, $F$ ואת $S$.

### לנבא

התחייבו לתשובה לכל אחד מארבעת הניבויים של דף המודול לפני שתמשיכו:

1. מה נותן הביטוי $\partial\ln Z/\partial\beta$, ובאיזה סימן?
2. חצו את הטמפרטורה של פרמגנט בשדה חלש. מה קורה למגנוט שלו?
3. הזרימו אנרגיה לתוך הפרמגנט מעבר לנקודה שבה מחצית מהמומנטים שלו מצביעים לכל כיוון. מה קורה
   לטמפרטורה שלו — ולאיזה כיוון יזרום אז החום בינו לבין גוף בטמפרטורת החדר?
4. כמה אנרגיה מחזיק מתנד הרמוני חם, $\kB T \gg \hbar\omega$? וכמה מחזיק מתנד קר?

**הניבויים שלכם:**

1.
2.
3.
4.

## חלק 2 — פונקציית החלוקה ביד

קחו את המערכת הדו-רמתית עם פער $\Delta = 2\kB T$. על נייר: רשמו את $Z$, אחר כך את $\ln Z$, ואז
גזרו לפי $\beta$ כדי לקבל את $U$. אחר כך הריצו את התא הבא, שעושה את אותם שלושה דברים — לפי
הנוסחה, ובאמצעות הפרש מרכזי של $\ln Z$ לפי $\beta$.

**מה שקיבלתם עבור $Z$ ועבור $U$:**

*(רשמו כאן לפני הרצת התא הבא)*

In [ ]:
temperature = QUANTUM / (2 * K_B)  # so that Delta = 2 k_B T
beta = 1 / (K_B * temperature)
z_by_hand = 1 + np.exp(-beta * QUANTUM)
print(f"Z by hand  = 1 + exp(-2)            = {z_by_hand:.6f}")
library_log_z = partition.log_z_two_level(QUANTUM, temperature)
print(f"ln Z       = {np.log(z_by_hand):.6f};  library: {library_log_z:.6f}")


def log_z_of_beta(b):
    return partition.log_z_two_level(QUANTUM, 1 / (K_B * b))


h = 1e-5 * beta
u_by_difference = -(log_z_of_beta(beta + h) - log_z_of_beta(beta - h)) / (2 * h)
u_by_formula = QUANTUM / (np.exp(2) + 1)
print(f"U = -d ln Z / d beta, by difference: {u_by_difference / QUANTUM:.8f} Delta")
print(f"U = Delta / (e^2 + 1), by formula:   {u_by_formula / QUANTUM:.8f} Delta")

שום דבר בהפרש לא ידע מהן הרמות או מה ההסתברות של כל אחת מהן. גזירת הלוגריתם של קבוע הנרמול
נתנה את האנרגיה הממוצעת.

## חלק 3 — מכונת ה-Z

הפונקציה `thermo_from_z` עושה אותו דבר לכל הגדלים בבת אחת: הפרשים מרכזיים לפי $\beta$ עבור $U$
(נגזרת ראשונה) ועבור $\sigma_E^2$ (נגזרת שנייה), ואחר כך

$$
C = \frac{\sigma_E^2}{\kB T^2}, \qquad F = -\kB T\ln Z, \qquad S = \frac{U - F}{T} .
$$

הריצו אותה על שלוש המערכות, מעשירית פער הרמות ועד פי עשרה ממנו, ושרטטו את השחזור (נקודות) מעל
הצורות הסגורות (קווים).

In [ ]:
N_SPINS = 10_000
systems = {
    "two-level": dict(
        log_z=lambda t: partition.log_z_two_level(QUANTUM, t),
        energy=lambda t: partition.two_level_energy(QUANTUM, t),
        capacity=lambda t: partition.two_level_heat_capacity(QUANTUM, t),
        gap=QUANTUM, n=1),
    "paramagnet": dict(
        log_z=lambda t: partition.log_z_paramagnet(N_SPINS, MU, FIELD, t),
        energy=lambda t: partition.paramagnet_energy(N_SPINS, MU, FIELD, t),
        capacity=lambda t: partition.paramagnet_heat_capacity(N_SPINS, MU, FIELD, t),
        gap=2 * MU * FIELD, n=N_SPINS),
    "oscillator": dict(
        log_z=lambda t: partition.log_z_harmonic(QUANTUM, t),
        energy=lambda t: partition.harmonic_energy(QUANTUM, t),
        capacity=lambda t: partition.harmonic_heat_capacity(QUANTUM, t),
        gap=QUANTUM, n=1),
}

fig, axes = plt.subplots(3, 3, figsize=(12, 8.5), sharex="col")
residuals = {}
for column, (name, spec) in enumerate(systems.items()):
    x = np.geomspace(0.1, 10.0, 400)  # k_B T / gap
    t = x * spec["gap"] / K_B
    rebuilt = partition.thermo_from_z(spec["log_z"], t)
    middle = (x >= 0.2) & (x <= 5.0)
    residuals[name] = (
        np.max(np.abs(rebuilt.energy / spec["energy"](t) - 1)),
        np.max(np.abs(rebuilt.heat_capacity[middle] / spec["capacity"](t[middle]) - 1)),
    )
    per = spec["gap"] * spec["n"]
    show = slice(None, None, 20)
    axes[0, column].semilogx(x, spec["energy"](t) / per, "k")
    axes[0, column].semilogx(x[show], rebuilt.energy[show] / per, "o", color="#2563eb", ms=4)
    axes[1, column].semilogx(x, rebuilt.entropy / (K_B * spec["n"]), "k")
    axes[1, column].semilogx(x[show], rebuilt.entropy[show] / (K_B * spec["n"]), "o",
                             color="#2563eb", ms=4)
    axes[2, column].semilogx(x, spec["capacity"](t) / (K_B * spec["n"]), "k")
    axes[2, column].semilogx(x[show], rebuilt.heat_capacity[show] / (K_B * spec["n"]), "o",
                             color="#2563eb", ms=4)
    axes[0, column].set_title(name)
    axes[2, column].set_xlabel("k_B T / gap")
axes[0, 0].set_ylabel("U / (N gap)")
axes[1, 0].set_ylabel("S / (N k_B)")
axes[2, 0].set_ylabel("C / (N k_B)")
plt.tight_layout()
plt.show()

for name, (u_err, c_err) in residuals.items():
    print(f"{name:<11} largest relative error: U {u_err:.1e}   "
          f"C (a fifth to five gaps) {c_err:.1e}")

### האם האנרגיה החופשית של מודול 10 היא באמת $-\kB T\ln Z$?

מודול 10 הגדיר $F = U - TS$ בלי שום צבר באופק. הטענה $F = -\kB T\ln Z$ היא משפט, ומשפט אפשר
לבחון. חשבו את האנרגיה החופשית של המתנד **פעמיים**, בלי ששום דבר יהיה משותף לשתי הדרכים:

- מתוך $\ln Z$: $F = -\kB T\ln Z$;
- מתוך ההסתברויות: $U = \sum_n E_n P_n$ ואנטרופיית גיבס $S = -\kB\sum_n P_n\ln P_n$,
  ואחר כך $F = U - TS$.

### לנבא

האם שתי התוצאות יתאימו בדיוק, בקירוב, או רק בטמפרטורה גבוהה?

**הניבוי שלכם:**

*(רשמו כאן לפני הרצת התא הבא)*

In [ ]:
print("   T [K]   levels   -k_B T ln Z [J]          U - T S (Gibbs) [J]      relative gap")
free_energy_gaps = []
for temperature in (20.0, QUANTUM / K_B, 300.0):
    n_max = partition.harmonic_cutoff(QUANTUM, temperature)
    sums = partition.level_sums(QUANTUM * (np.arange(n_max + 1) + 0.5), temperature)
    gap = relative_error(sums.free_energy_direct, sums.free_energy_from_z)
    free_energy_gaps.append(gap)
    print(f"{temperature:8.2f}   {n_max + 1:6d}   {sums.free_energy_from_z:+.15e}   "
          f"{sums.free_energy_direct:+.15e}   {gap:.1e}")

# The falsifying experiment for "F = -k_B T ln Z is just a definition": two independent
# computations, one of them knowing nothing of Z, agree to the last digit a float can hold.
assert max(free_energy_gaps) < 1e-13

שתי התוצאות מתאימות עד כדי שגיאת עיגול. הגודל $-\kB T\ln Z$ אינו גודל חדש שבמקרה חולק שם; הוא
*הוא עצמו* הביטוי $U - TS$ של מודול 10, כאשר $S$ היא האנטרופיה שההסתברויות נושאות.

## חלק 4 — כמה קטן צריך להיות הצעד?

להפרשים המרכזיים יש **שגיאת קטיעה** שגודלה כמו $h^2$: חצו את הצעד והיא קטנה פי ארבעה. אבל
הנוסחאות מחסרות ערכים של $\ln Z$ שמסכימים ביותר ויותר ספרות ככל שהצעד $h$ מתכווץ, ומספר נקודה
צפה מחזיק רק שש-עשרה ספרות. זו **שגיאת עיגול**, והיא גדלה ככל שהצעד מתכווץ — כמו $1/h$ עבור $U$,
וכמו $1/h^2$ עבור $C$.

### לנבא

ככל שהצעד היחסי $h/\beta$ יורד מהערך $10^{-1}$ עד $10^{-10}$, שרטטו את השגיאה של $U$
ואת השגיאה של $C$ על גרף לוג-לוג. היכן כל אחת מהן קטנה ביותר?

**הסקיצה שלכם, במילים:**

*(רשמו כאן לפני הרצת התא הבא)*

In [ ]:
t_one = np.array([QUANTUM / K_B])  # k_B T = hbar omega
exact_u = partition.harmonic_energy(QUANTUM, t_one[0])
exact_c = partition.harmonic_heat_capacity(QUANTUM, t_one[0])
oscillator_log_z = systems["oscillator"]["log_z"]

relative_steps = np.geomspace(1e-1, 1e-10, 37)
u_errors, c_errors = [], []
for step in relative_steps:
    rebuilt = partition.thermo_from_z(oscillator_log_z, t_one, rel_step=step)
    u_errors.append(relative_error(rebuilt.energy[0], exact_u))
    c_errors.append(relative_error(rebuilt.heat_capacity[0], exact_c))

fig, ax = plt.subplots(figsize=(7, 4.2))
ax.loglog(relative_steps, u_errors, "o-", color="#2563eb", ms=4, label="U")
ax.loglog(relative_steps, c_errors, "s-", color="#dc2626", ms=4, label="C")
ax.axvline(partition.FIRST_DERIVATIVE_REL_STEP, color="#2563eb", ls=":", lw=1)
ax.axvline(partition.SECOND_DERIVATIVE_REL_STEP, color="#dc2626", ls=":", lw=1)
ax.set_xlabel("h / beta")
ax.set_ylabel("relative error")
ax.invert_xaxis()
ax.legend()
plt.tight_layout()
plt.show()

print(f"default steps: {partition.FIRST_DERIVATIVE_REL_STEP:.1e} (U), "
      f"{partition.SECOND_DERIVATIVE_REL_STEP:.1e} (C)")
for label, errors in (("U", u_errors), ("C", c_errors)):
    best = relative_steps[np.argmin(errors)]
    print(f"smallest error of {label}: {min(errors):.1e} at h/beta = {best:.0e}")

הקווים המנוקדים הם ברירות המחדל שהמכונה משתמשת בהן: $\epsilon^{1/3}$ עבור $U$ ואילו $\epsilon^{1/4}$
עבור $C$, כאשר $\epsilon = 2.2\times10^{-16}$ — גדלי הצעד שמאזנים בין שתי השגיאות. מימין לכל
מינימום השגיאה יורדת בשיפוע $2$; משמאלו היא עולה.

עכשיו מדדו את סדר ההתכנסות בתחום שבו שגיאת הקטיעה שולטת, עם פס שגיאה — זו המדידה של המעבדה.

In [ ]:
refinements = [10, 20, 40, 80, 160]


def rebuilt_at(n):
    return partition.thermo_from_z(oscillator_log_z, t_one, rel_step=1.0 / n)


orders = {}
for label, attribute, exact in (("U", "energy", exact_u), ("C", "heat_capacity", exact_c)):
    study = convergence_study(lambda n, a=attribute: float(getattr(rebuilt_at(n), a)[0]),
                              refinements, exact)
    (slope, _), cov = np.polyfit(np.log(study.refinements), np.log(study.errors), 1, cov=True)
    orders[label] = (-slope, float(np.sqrt(cov[0, 0])))
    ratios = study.errors[:-1] / study.errors[1:]
    print(f"{label}: order p = {-slope:.4f} +/- {np.sqrt(cov[0, 0]):.4f}   "
          f"(each halving divides the error by {ratios.min():.3f} to {ratios.max():.3f})")

## חלק 5 — הפרמגנט

לפנינו $N$ מומנטים בלתי תלויים, כל אחד באנרגיה $-\mu B$ לאורך השדה או $+\mu B$ נגדו. למומנט אחד
יש $z = 2\cosh(\mu B/\kB T)$, ואי-התלות נותנת $\ln Z_N = N\ln z$ — שום סכום על פני $2^N$
מיקרו-מצבים אינו מחושב לעולם. המגנוט הוא $M = N\mu\tanh(\mu B/\kB T)$.

### לנבא

בשדה של $1\ \mathrm{T}$, פי כמה משתנה $M$ כשעוברים מהטמפרטורה $300\ \mathrm{K}$ אל
$150\ \mathrm{K}$? ומהטמפרטורה $1\ \mathrm{K}$ אל $0.5\ \mathrm{K}$?

**הניבוי שלכם:**

*(רשמו כאן לפני הרצת התא הבא)*

In [ ]:
additivity = (partition.log_z_paramagnet(4096, MU, FIELD, 2.0)
              / partition.log_z_paramagnet(1, MU, FIELD, 2.0))
print(f"additivity: ln Z(4096 moments) / ln Z(1 moment) = {additivity:.10f}")

pairs = [(300.0, 150.0), (1.0, 0.5), (0.2, 0.1)]
for hot, cold in pairs:
    ratio = (partition.paramagnet_magnetization(N_SPINS, MU, FIELD, cold)
             / partition.paramagnet_magnetization(N_SPINS, MU, FIELD, hot))
    print(f"M({cold} K) / M({hot} K) = {ratio:.5f}")

# Curie's law, fitted: in the weak-field regime M is proportional to 1/T; the slope is the
# Curie constant N mu^2 B / k_B.
curie_t = np.linspace(50.0, 300.0, 11)
curie_m = partition.paramagnet_magnetization(N_SPINS, MU, FIELD, curie_t)
(curie_slope, curie_intercept), curie_cov = np.polyfit(1 / curie_t, curie_m, 1, cov=True)
curie_expected = N_SPINS * MU**2 * FIELD / K_B
print(f"\nfitted Curie constant: {curie_slope:.5e} +/- {np.sqrt(curie_cov[0, 0]):.1e} J K/T")
print(f"N mu^2 B / k_B:        {curie_expected:.5e} J K/T")

x = np.geomspace(0.01, 10, 300)  # mu B / (k_B T)
fig, (left, right) = plt.subplots(1, 2, figsize=(11, 3.8))
left.semilogx(x, np.tanh(x), "k", label="N mu tanh(mu B / k_B T)")
left.semilogx(x, x, "--", color="#2563eb", label="Curie law")
left.axhline(1.0, color="grey", ls=":")
left.set_ylim(0, 1.2)
left.set_xlabel("mu B / (k_B T)")
left.set_ylabel("M / (N mu)")
left.legend()
right.plot(1 / curie_t, curie_m / (N_SPINS * MU), "o", color="#2563eb")
right.plot(1 / curie_t, (curie_intercept + curie_slope / curie_t) / (N_SPINS * MU), "k-")
right.set_xlabel("1 / T  [1/K]")
right.set_ylabel("M / (N mu)")
right.set_title("Curie regime: M against 1/T")
plt.tight_layout()
plt.show()

### הפלוקטואציות של האנרגיה

הנגזרת השנייה של $\ln Z_N$ היא השונות של האנרגיה. אי-התלות גורמת לשני הגדלים $U$
וגם $\sigma_E^2$ לגדול כמו $N$, ולכן היחס $\sigma_E/|U|$ אמור לרדת כמו $N^{-1/2}$. מדדו את
המעריך מתוך הנגזרות השניות של המכונה עצמה — בלי שום צורה סגורה.

In [ ]:
sizes = [10, 100, 1_000, 10_000, 100_000, 1_000_000]
relative_spread = []
for n in sizes:
    rebuilt = partition.thermo_from_z(
        lambda t, n=n: partition.log_z_paramagnet(n, MU, FIELD, t), np.array([1.0]))
    relative_spread.append(float(np.sqrt(rebuilt.energy_variance[0]) / abs(rebuilt.energy[0])))
    print(f"N = {n:>9,d}   sigma_E / |U| = {relative_spread[-1]:.6f}")
fluctuation_exponent = scaling_exponent(sizes, relative_spread)
print(f"\nfitted exponent: {fluctuation_exponent:.7f}  (the derivation says -1/2)")

## חלק 6 — הפרמגנט באנרגיה קבועה, וטמפרטורה שלילית

בודדו את הפרמגנט. כאשר $n$ מתוך $N$ המומנטים שלו הפוכים, $U = \mu B(2n - N)$
וכן $S = \kB\ln\binom{N}{n}$ — שוב המטבעות של מודול 08. הטמפרטורה היא השיפוע של מודול 09,
$1/T = \partial S/\partial U$.

### לנבא

שרטטו את $S(U)$ בתחום שבין $U = -N\mu B$ לבין $+N\mu B$. היכן השיפוע שלה אפס? מה הסימן של $T$
מעבר לנקודה הזו?

**הסקיצה שלכם, במילים:**

*(רשמו כאן לפני הרצת התא הבא)*

In [ ]:
spins = partition.spin_entropy_of_energy(100, MU, FIELD)
beta_dome = spins.beta()
fig, (left, right) = plt.subplots(1, 2, figsize=(11, 3.8))
left.plot(spins.energy / (MU * FIELD), spins.entropy / K_B, "k")
left.set_xlabel("U / (mu B)")
left.set_ylabel("S / k_B")
right.plot(spins.energy / (MU * FIELD), beta_dome * MU * FIELD, "o", color="#2563eb", ms=3)
right.axhline(0, color="grey", lw=1)
right.set_xlabel("U / (mu B)")
right.set_ylabel("beta mu B")
plt.tight_layout()
plt.show()

print("reversed   U/(mu B)   S/k_B     beta mu B    T [K]")
for n in (10, 30, 49, 50, 51, 70, 85):
    b = beta_dome[n]
    t_text = "  +/-inf" if b == 0 else f"{1 / (K_B * b):+8.3f}"
    print(f"{n:8d}   {spins.energy[n] / (MU * FIELD):+8.0f}   {spins.entropy[n] / K_B:7.3f}"
          f"   {b * MU * FIELD:+9.4f}   {t_text}")

### לאן הולך החום?

הכינו את הפרמגנט כאשר $85$ מתוך $100$ המומנטים שלו הפוכים: $T \approx -0.79\ \mathrm{K}$.
הצמידו אותו למוצק איינשטיין של $200$ מתנדים שהקוונט שלו הוא היפוך אחד, $2\mu B$, ומנו כל דרך
שבה הזוג יכול לחלוק את האנרגיה שלו. נסו את המוצק קר, חמים וחם מאוד.

### לנבא

עבור כל טמפרטורה של המוצק, האם אנרגיה זורמת אל תוך הפרמגנט או החוצה ממנו?

**הניבוי שלכם:**

*(רשמו כאן לפני הרצת התא הבא)*

In [ ]:
N_OSC = 200
flip = 2 * MU * FIELD
print(" solid quanta/osc   solid T [K]   reversed: start -> most probable   quanta to solid"
      "   dlnOmega(one quantum out)   (one quantum in)")
contacts = {}
for per_oscillator in (0.1, 1.0, 10.0, 1000.0):
    contact = partition.spin_solid_contact(100, 85, N_OSC, int(per_oscillator * N_OSC))
    contacts[per_oscillator] = contact
    solid_t = partition.harmonic_temperature(per_oscillator * flip, flip)
    print(f"{per_oscillator:16.1f}   {solid_t:11.2f}   {contact.initial_flipped:8d} -> "
          f"{contact.most_probable_flipped:3d}              {contact.energy_to_solid:5d}"
          f"            {contact.log_omega_change(-1):+.3f}                   "
          f"{contact.log_omega_change(+1):+.3f}")

# The falsifying experiment for "negative temperature is colder than absolute zero": at every
# solid temperature, total multiplicity rises only when the inverted spins LOSE energy.
for contact in contacts.values():
    assert contact.energy_to_solid > 0
    assert contact.log_omega_change(-1) > 0 > contact.log_omega_change(+1)

אנרגיה עוזבת את הפרמגנט בכל פעם — גם כשהמוצק נמצא ביותר מאלף קלווין. טמפרטורה שלילית חמה יותר
מכל טמפרטורה חיובית: בסידור לפי $-\beta$, היא יושבת מעבר לנקודה $T = +\infty$, ולא מתחת
לנקודה $T = 0$.

## חלק 7 — המתנד ההרמוני

הסולם $\hbar\omega(n + 1/2)$ הוא טור הנדסי, שסכומו $z = e^{-x/2}/(1 - e^{-x})$ כאשר
$x = \hbar\omega/\kB T$. בדקו תחילה את הצורה הסגורה מול הסכום המפורש על פני הרמות, הנקטע במקום
שבו גורמי בולצמן כבר דעכו. אחר כך הביטו בשני הקצוות של תחום הטמפרטורות.

In [ ]:
print("  T [K]    n_max    ln z (sum)            ln z (closed form)")
for temperature in (5.0, 72.43, 2000.0):
    n_max = partition.harmonic_cutoff(QUANTUM, temperature)
    summed = partition.log_z_harmonic(QUANTUM, temperature, n_max=n_max)
    closed = partition.log_z_harmonic(QUANTUM, temperature)
    print(f"{temperature:7.2f}   {n_max:6d}   {summed:.15f}   {closed:.15f}")

x = np.geomspace(0.05, 50, 400)  # k_B T / (hbar omega)
t = x * QUANTUM / K_B
fig, (left, right) = plt.subplots(1, 2, figsize=(11, 3.8))
left.loglog(x, partition.harmonic_energy(QUANTUM, t, zero_point=False) / QUANTUM, "k",
            label="U above the ground state")
left.loglog(x, x, "--", color="#2563eb", label="equipartition, k_B T")
left.set_xlabel("k_B T / (hbar omega)")
left.set_ylabel("U / (hbar omega)")
left.legend()
right.semilogx(x, partition.harmonic_heat_capacity(QUANTUM, t) / K_B, "k")
right.axhline(1.0, color="#2563eb", ls="--")
right.set_xlabel("k_B T / (hbar omega)")
right.set_ylabel("C / k_B")
plt.tight_layout()
plt.show()

for ratio in (100.0, 10.0, 1.0, 0.2, 0.1):
    temperature = ratio * QUANTUM / K_B
    above_ground = partition.harmonic_energy(QUANTUM, temperature, zero_point=False)
    capacity = partition.harmonic_heat_capacity(QUANTUM, temperature)
    print(f"k_B T = {ratio:5.1f} hbar omega:  U/k_B T (above ground) = "
          f"{above_ground / (K_B * temperature):.4f}   C/k_B = {capacity / K_B:.4f}")

כשהוא חם, המתנד מחזיק $\kB T$ — חלוקת האנרגיה השווה, הכלל שמודולים 04 ו-06 הניחו, מתקבלת
כעת כגבול. כשהוא קר, קיבול החום שלו דועך מעריכית: הוא יכול לקבל אנרגיה רק קוונט שלם בכל פעם.

## חלק 8 — המוצק של מודול 01, בקריאה חדשה

מודול 01 דימה שני מוצקי איינשטיין המחליפים קוונטים, וקרא את הטמפרטורה של כל אחד מהם לפי חלוקת
האנרגיה השווה, $T = U/(n\kB)$. מפרט המודל שלו הזהיר שהקריאה הזו נכשלת כשהמוצק קר. פונקציית
החלוקה של המתנד נותנת את הקריאה המדויקת, $T = \hbar\omega/[\kB\ln(1 + n\hbar\omega/U)]$.

הריצו את הסימולציה של מודול 01 בכמה אנרגיות, עם שמונה זרעים לכל אחת, וקראו את נקודות הקצה בשתי
הדרכים.

### לנבא

כשיש $0.1$ קוונטים לכל מתנד, האם הקריאה של מודול 01 תהיה חמה מדי, קרה מדי, או נכונה?

**הניבוי שלכם:**

*(רשמו כאן לפני הרצת התא הבא)*

In [ ]:
N_BODY = 40


def endpoint_temperature(rng, state):
    # Body A's temperature, read through Z, averaged over the second half of the run.
    run = equilibrium.simulate_energy_exchange(state, 20 * state.total_quanta, rng)
    tail = run.q_a[len(run.q_a) // 2:]
    return float(partition.harmonic_temperature(tail.mean() / N_BODY * QUANTUM, QUANTUM))


print("quanta/osc   module 01 reads [K]   Z predicts [K]   simulated, read by Z [K]")
overlay = []
for per_oscillator in (0.1, 0.25, 0.5, 1.0, 2.0, 4.0, 8.0):
    t_equipartition = per_oscillator * QUANTUM / K_B
    state = equilibrium.from_temperatures(N_BODY, N_BODY, 1.6 * t_equipartition,
                                          0.4 * t_equipartition, QUANTUM)
    study = seed_study(lambda rng, s=state: endpoint_temperature(rng, s), n_seeds=8)
    shared = state.total_quanta / state.total_oscillators
    predicted = partition.harmonic_temperature(shared * QUANTUM, QUANTUM)
    overlay.append((shared, predicted, study))
    print(f"{shared:9.3f}   {shared * QUANTUM / K_B:18.1f}   {predicted:14.1f}   "
          f"{study.mean:10.1f} +/- {study.standard_error:.1f}")

shared_u = np.array([row[0] for row in overlay])
fig, ax = plt.subplots(figsize=(7, 4.5))
x = np.geomspace(0.05, 20, 300)
ax.loglog(x, 1 / np.expm1(1 / x), "k", label="oscillator Z")
ax.loglog(x, x, "--", color="grey", label="equipartition (module 01)")
ax.errorbar([row[2].mean * K_B / QUANTUM for row in overlay], shared_u,
            xerr=[row[2].standard_error * K_B / QUANTUM for row in overlay],
            fmt="o", color="#2563eb", label="simulated endpoints, read by Z")
ax.plot(shared_u, shared_u, "o", mfc="none", color="grey",
        label="same endpoints, read by module 01")
ax.set_xlabel("k_B T / (hbar omega)")
ax.set_ylabel("quanta per oscillator")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

הסימולציה מספקת את האנרגיות; פונקציית החלוקה מספקת את המדחום. כשהמוצק חם, שתי הקריאות נבדלות
בחצי קוונט. כשיש $0.1$ קוונטים לכל מתנד, הקריאה של מודול 01 קרה פי ארבעה מן האמת — בדיוק התחום
שמפרט המודל שלו החריג. ההשוואה ממחישה; שלב 7 של הגזירה הוא שמבסס.

## חלק 9 — בדיקות אוטומטיות

חישוב שלא בדקתם הוא תמונה, לא ראיה. אלה אותן טענות בדיקה שרצות בחבילת הבדיקות של הפרויקט.

In [ ]:
# 1. By hand and by difference, U of the two-level system at Delta = 2 k_B T (Part 2).
assert relative_error(u_by_difference, u_by_formula) < 1e-8

# 2. The Z-machine reproduces every closed form (Part 3) ...
for u_err, c_err in residuals.values():
    assert u_err < 1e-6 and c_err < 1e-4
# ... and -k_B T ln Z is U - TS with the Gibbs entropy (checked above).

# 3. The stencil is second order (Part 4).
for order, error in orders.values():
    assert abs(order - 2.0) < 5 * error + 0.01

# 4. Curie's law: halving T in a weak field doubles M; the fitted constant is N mu^2 B / k_B.
assert relative_error(curie_slope, curie_expected) < 1e-3
# 5. Relative fluctuations fall as N^(-1/2) (Part 5).
assert abs(fluctuation_exponent + 0.5) < 1e-4

# 6. The dome's slope changes sign at the top, and inverted spins give energy away (Part 6).
assert beta_dome[50] == 0 and beta_dome[49] > 0 > beta_dome[51]

# 7. Every simulated endpoint agrees with Z's temperature, within three standard errors (Part 8).
for _, predicted, study in overlay:
    assert study.agrees_with(predicted)

print("all checks passed")

## חלק 10 — חקרו בעצמכם

בחרו מערכת ואת סקאלת הרמות שלה ביחידות של טמפרטורת ייחוס $T_0 = 100\ \mathrm{K}$,
ואז לחצו **Run Interact**. הלוחות מציגים את $U$, $S$, $F$ ואת $C$ כשהם משוחזרים מתוך $\ln Z$
(נקודות) מעל העקומות המדויקות (קווים), ואת השגיאה היחסית הגדולה ביותר של כל אחד. שלושה דברים
שכדאי לנסות:

1. הגדילו את הקוונט של המתנד. היכן על ציר הטמפרטורה נדלק $C$?
2. תנו לפרמגנט מומנט יחיד, ואחר כך עשרת אלפים. מה משתנה בגרפים, ומה לא, לאחר שכל גודל מחולק
   במספר $N$?
3. דחפו את סקאלת הרמות לקצה המחוון. באיזה לוח השחזור מראה לראשונה שגיאה הנראית לעין, ומדוע
   דווקא בו?

In [ ]:
import ipywidgets as widgets

T_REFERENCE = 100.0  # K


def explore(system="oscillator", scale=1.0, log10_spins=4.0):
    level = scale * K_B * T_REFERENCE
    n = int(round(10**log10_spins))
    if system == "two-level":
        log_z = lambda t: partition.log_z_two_level(level, t)  # noqa: E731
        exact = (partition.two_level_energy(level, temps), None,
                 partition.two_level_heat_capacity(level, temps))
        n = 1
    elif system == "paramagnet":
        moment = level / FIELD
        log_z = lambda t: partition.log_z_paramagnet(n, moment, FIELD, t)  # noqa: E731
        exact = (partition.paramagnet_energy(n, moment, FIELD, temps),
                 partition.paramagnet_entropy(n, moment, FIELD, temps),
                 partition.paramagnet_heat_capacity(n, moment, FIELD, temps))
    else:
        log_z = lambda t: partition.log_z_harmonic(level, t)  # noqa: E731
        exact = (partition.harmonic_energy(level, temps), partition.harmonic_entropy(level, temps),
                 partition.harmonic_heat_capacity(level, temps))
        n = 1
    rebuilt = partition.thermo_from_z(log_z, temps)
    fig, axes = plt.subplots(1, 4, figsize=(14, 3.2))
    series = (("U / (N k_B T0)", rebuilt.energy, exact[0], K_B * T_REFERENCE),
              ("S / (N k_B)", rebuilt.entropy, exact[1], K_B),
              ("F / (N k_B T0)", rebuilt.free_energy, None, K_B * T_REFERENCE),
              ("C / (N k_B)", rebuilt.heat_capacity, exact[2], K_B))
    for ax, (label, value, reference, unit) in zip(axes, series, strict=True):
        if reference is not None:
            ax.semilogx(temps, reference / (n * unit), "k")
            worst = np.max(np.abs(value / reference - 1))
            ax.set_title(f"max rel. error {worst:.0e}", fontsize=9)
        ax.semilogx(temps[::15], value[::15] / (n * unit), "o", color="#2563eb", ms=3)
        ax.set_xlabel("T [K]")
        ax.set_ylabel(label)
    plt.tight_layout()
    plt.show()


temps = np.geomspace(1.0, 10_000.0, 300)
widgets.interact_manual(
    explore,
    system=["oscillator", "two-level", "paramagnet"],
    scale=widgets.FloatLogSlider(value=1.0, base=10, min=-1, max=1, step=0.05),
    log10_spins=widgets.FloatSlider(value=4.0, min=0.0, max=4.0, step=0.5),
)

## בחנו את הבנתכם

הריצו את התא שלהלן לחידון הנבדק אוטומטית. אותן שאלות, עם הסברים כתובים לכל אפשרות, נמצאות
בדף המודול.

In [ ]:
import json
from pathlib import Path

quiz_path = Path("..") / "_quiz" / "12-partition-functions.json"
if quiz_path.exists():
    from jupyterquiz import display_quiz

    # Parsed here with an explicit encoding: jupyterquiz opens the file with the platform
    # default, which cannot decode the Hebrew edition of this notebook on Windows.
    display_quiz(json.loads(quiz_path.read_text(encoding="utf-8")))
else:
    print("Quiz not generated yet — run: uv run python scripts/render_quizzes.py")

## לפני שאתם עוזבים

רשמו כמה משפטים על כל אחת, בתא שלהלן.

1. מה ניבאתם שהתברר כשגוי, ומה בדיוק היה הפגם בנימוקכם?
2. מכונת ה-Z מעולם לא ראתה רמה או הסתברות. הסבירו, בלי משוואות, כיצד היא מצאה את קיבול החום.
3. בחלק 4 צעד קטן יותר עשה את התשובה *גרועה* יותר. מדוע, ומדוע $C$ נפגע לפני $U$?
4. בחלק 6 פרמגנט בטמפרטורה $-0.79\ \mathrm{K}$ חימם מוצק בטמפרטורה $1344\ \mathrm{K}$. הסבירו
   זאת למי שמאמין שטמפרטורות שליליות נמצאות מתחת לאפס המוחלט.

**התשובות שלכם:**

1.
2.
3.
4.